## Summary (Current Implementation)

### What Is Implemented In The App

The production parser and converters now support a broader `.note` workflow than this notebook originally documented.

1. `.note` parsing (`OnyxNoteParser.kt`)
- Native `.note` ZIP reading with active-entry filtering.
- Multi-page grouping (`NoteDocument.pages`) with per-page stroke extraction.
- Shape/point matching and stash/archived entry handling.
- Chunk-table point decoding with fallback parser paths.

2. OCR + searchable PDF
- Handwriting recognition via Onyx MyScript (`OnyxHWREngine`).
- Detailed OCR parsing includes word-level bounding boxes when available.
- Searchable PDF text layer placement uses OCR geometry first, with heuristics fallback.

3. Batch converter
- Converts multi-page `.note` to markdown and optional searchable PDF.
- SAF output support (external storage/document providers).
- Stage profiling and bottleneck logging were added during optimization.

4. Capture (Inbox) sync
- Obsidian markdown export from inbox capture.
- Optional PDF generation integrated with capture flow.
- Pagination-aware capture PDF export and A5 output support.

5. Obsidian settings and templates
- Dedicated Obsidian tab in settings.
- Shared options for capture and batch output.
- Optional markdown template file selector (SAF URI).
- Template-aware markdown rendering with auto-fill for YAML keys: `created`, `modified`.

### Source Of Truth

For current behavior, use code as source of truth:
- `app/src/main/java/com/ethran/notable/io/OnyxNoteParser.kt`
- `app/src/main/java/com/ethran/notable/io/OnyxHWREngine.kt`
- `app/src/main/java/com/ethran/notable/io/SearchablePdfGenerator.kt`
- `app/src/main/java/com/ethran/notable/noteconverter/ObsidianConverter.kt`
- `app/src/main/java/com/ethran/notable/io/InboxSyncEngine.kt`
- `app/src/main/java/com/ethran/notable/ui/components/GeneralSettings.kt`

### Note

Some exploratory cells below remain historical/research-oriented and may not reflect latest production internals.

### Implementation Status (Updated)

Current implementation is already integrated in Notable and has moved beyond the initial standalone concept.

**Current key files:**
1. `app/src/main/java/com/ethran/notable/io/OnyxNoteParser.kt`
2. `app/src/main/java/com/ethran/notable/io/OnyxHWREngine.kt`
3. `app/src/main/java/com/ethran/notable/io/SearchablePdfGenerator.kt`
4. `app/src/main/java/com/ethran/notable/noteconverter/ObsidianConverter.kt`
5. `app/src/main/java/com/ethran/notable/io/InboxSyncEngine.kt`

**Implemented capabilities:**
- Parse `.note` archives with multi-page support.
- Run HWR and produce markdown output.
- Generate searchable PDFs for batch and capture workflows.
- Support SAF-based output directories for markdown/PDF.
- Use optional Obsidian markdown templates for both capture and batch output.

**Practical guidance:**
Use this notebook as reverse-engineering notes. For app behavior and data contracts, rely on the Kotlin source files listed above.

In [ ]:
package com.ethran.noteconverter.io

import java.io.InputStream
import java.nio.ByteBuffer
import java.nio.ByteOrder
import java.util.zip.ZipInputStream
import org.json.JSONObject

data class StrokePoint(
    val x: Float,
    val y: Float,
    val pressure: Float,
    val dt: Long  // Delta time in milliseconds
)

data class ParsedStroke(
    val id: String,
    val points: List<StrokePoint>,
    val penType: Int = 2,  // Default: ballpoint
    val penColor: Int = -16777216  // Default: black
)

data class NoteDocument(
    val title: String,
    val pageWidth: Float,
    val pageHeight: Float,
    val strokes: List<ParsedStroke>
)

object OnyxNoteParser {
    
    /**
     * Parse an Onyx .note file from an InputStream.
  

### Kotlin: OnyxNoteParser.kt

Parse `.note` ZIP archive and extract strokes.

---

## Part 2: Android Application Implementation

Now let's create the Kotlin code for a standalone Android app that:
1. Parses `.note` files  
2. Calls `OnyxHWREngine` for recognition
3. Generates markdown files

### Architecture

```
NoteConverterApp/
├── io/
│   ├── OnyxNoteParser.kt      # Parse .note files
│   ├── OnyxHWREngine.kt       # Reuse from Notable
│   └── MarkdownGenerator.kt   # Convert recognized text to .md
├── ui/
│   └── ConverterScreen.kt     # File picker + progress UI
└── MainActivity.kt
```

In [ ]:
import datetime

def convert_to_notable_stroke_format(strokes: List[Dict], page_width: float = 1860, page_height: float = 2480):
    """Convert parsed strokes to Notable database format."""
    notable_strokes = []
    
    for stroke in strokes:
        if not stroke['points']:
            continue
        
        # Notable expects points with: x, y, pressure, dt (delta time in ms)
        points_list = []
        base_timestamp = int(datetime.datetime.now().timestamp() * 1000)
        
        for i, pt in enumerate(stroke['points']):
            point_obj = {
                'x': pt['x'],
                'y': pt['y'],
                'pressure': pt['pressure'],
                'dt': pt.get('timestamp_delta', i * 10)  # Fallback to 10ms intervals
            }
            points_list.append(point_obj)
        
        notable_stroke = {
            'id': stroke['id'] or f"stroke-{len(notable_strokes)}",
            'pageId': 'imported-page',
            'createdAt': base_timestamp,
            'points': points_list,
            'penType': 2,  # Ballpoint pen
            'penColor': -16777216,  # Black
            'penWidth': 4.7244096
        }
        
        notable_strokes.append(notable_stroke)
    
    return notable_strokes

# Convert our strokes
notable_format = convert_to_notable_stroke_format(individual_strokes)

print("=== Notable Format ===\n")
for i, stroke in enumerate(notable_format, 1):
    print(f"Stroke {i}:")
    print(f"  ID: {stroke['id']}")
    print(f"  Points: {len(stroke['points'])}")
    print(f"  Pen: Type={stroke['penType']}, Width={stroke['penWidth']:.2f}")
    print(f"  Sample points: {stroke['points'][:2]}")
    print()

## Step 7: Convert to MyScript-Compatible Format

The existing `OnyxHWREngine.kt` expects strokes in the Notable database format. We need to convert our parsed points to match that structure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = ['blue', 'green', 'red']

for idx, (stroke, ax, color) in enumerate(zip(individual_strokes, axes, colors), 1):
    if not stroke['points']:
        continue
    
    xs = [p['x'] for p in stroke['points']]
    ys = [p['y'] for p in stroke['points']]
    pressures = [p['pressure'] for p in stroke['points']]
    
    # Plot stroke path with pressure-based alpha
    for i in range(len(xs) - 1):
        ax.plot(xs[i:i+2], ys[i:i+2], 
                color=color, 
                alpha=0.3 + 0.7 * pressures[i],
                linewidth=2)
    
    # Draw bounding box
    bbox = stroke['bbox']
    rect = patches.Rectangle(
        (bbox['left'], bbox['top']),
        bbox['right'] - bbox['left'],
        bbox['bottom'] - bbox['top'],
        linewidth=1, edgecolor='gray', facecolor='none', linestyle='--'
    )
    ax.add_patch(rect)
    
    # Invert Y axis (drawing coordinates start from top-left)
    ax.invert_yaxis()
    ax.set_aspect('equal')
    ax.set_title(f'Stroke {idx} ({len(stroke["points"])} points)')
    ax.set_xlabel('X coordinate')
    ax.set_ylabel('Y coordinate')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Visualize the Three Strokes

In [ ]:
def assign_points_to_strokes(all_points: List[Dict], strokes_meta: List[Dict]) -> List[Dict]:
    """Assign points to strokes based on bounding boxes."""
    strokes = []
    
    for meta in strokes_meta:
        bbox = meta['bbox']
        left, top, right, bottom = bbox['left'], bbox['top'], bbox['right'], bbox['bottom']
        
        # Find points within this bounding box (with small tolerance)
        tolerance = 5.0
        stroke_points = []
        for pt in all_points:
            if (left - tolerance <= pt['x'] <= right + tolerance and 
                top - tolerance <= pt['y'] <= bottom + tolerance):
                stroke_points.append(pt)
        
        # Sort by index to maintain stroke order
        stroke_points.sort(key=lambda p: p['index'])
        
        strokes.append({
            'id': meta['id'],
            'bbox': bbox,
            'points': stroke_points,
            'point_count': len(stroke_points)
        })
    
    return strokes

# Combine metadata with points
individual_strokes = assign_points_to_strokes(parsed_points['points'], strokes_meta)

print(f"=== Stroke Summary ===\n")
for i, stroke in enumerate(individual_strokes, 1):
    print(f"Stroke {i}: {stroke['point_count']} points")
    if stroke['points']:
        min_x = min(p['x'] for p in stroke['points'])
        max_x = max(p['x'] for p in stroke['points'])
        min_y = min(p['y'] for p in stroke['points'])
        max_y = max(p['y'] for p in stroke['points'])
        print(f"  Range: X=[{min_x:.1f}, {max_x:.1f}], Y=[{min_y:.1f}, {max_y:.1f}]")
        avg_pressure = sum(p['pressure'] for p in stroke['points']) / len(stroke['points'])
        print(f"  Avg Pressure: {avg_pressure:.3f}")
    print()

## Step 5: Separate Points into Individual Strokes

Use bounding boxes from shape metadata to cluster points into separate strokes.

In [ ]:
def parse_points_file(data: bytes) -> Dict[str, any]:
    """Parse binary points file to extract stroke coordinates."""
    stream = BytesIO(data)
    
    # Read header
    version = struct.unpack('>I', stream.read(4))[0]
    
    # Read page ID (ASCII, null-terminated or space-padded)
    page_id_bytes = stream.read(48)
    page_id = page_id_bytes.decode('ascii').rstrip('\x00 ')
    
    # Read points collection ID (UUID format)
    points_id_bytes = stream.read(36)
    points_id = points_id_bytes.decode('ascii')
    
    # Skip padding
    stream.read(4)
    
    # Parse point data
    points = []
    while True:
        try:
            # Read X coordinate (float32, big-endian)
            x_bytes = stream.read(4)
            if len(x_bytes) < 4:
                break
            x = struct.unpack('>f', x_bytes)[0]
            
            # Read Y coordinate (float32, big-endian)
            y = struct.unpack('>f', stream.read(4))[0]
            
            # Read timestamp delta (variable length, appears to be 2-3 bytes)
            ts_bytes = stream.read(3)
            timestamp_delta = int.from_bytes(ts_bytes, 'big')
            
            # Read pressure (2 bytes)
            pressure = struct.unpack('>H', stream.read(2))[0] / 4095.0  # Normalize to 0-1
            
            # Read index (4 bytes, sequence number)
            index = struct.unpack('>I', stream.read(4))[0]
            
            points.append({
                'x': x,
                'y': y,
                'pressure': pressure,
                'timestamp_delta': timestamp_delta,
                'index': index
            })
            
        except struct.error:
            break
    
    return {
        'version': version,
        'page_id': page_id,
        'points_id': points_id,
        'points': points
    }

with zipfile.ZipFile(note_file, 'r') as zf:
    # Find points files
    points_files = [n for n in zf.namelist() if '/point/' in n and n.endswith('#points')]
    
    if points_files:
        points_path = points_files[0]
        points_data = zf.read(points_path)
        parsed_points = parse_points_file(points_data)
        
        print(f"=== Points File ===")
        print(f"Version: {parsed_points['version']}")
        print(f"Page ID: {parsed_points['page_id']}")
        print(f"Points Collection ID: {parsed_points['points_id']}")
        print(f"Total Points: {len(parsed_points['points'])}")
        print(f"\nFirst 5 points:")
        for p in parsed_points['points'][:5]:
            print(f"  ({p['x']:7.2f}, {p['y']:7.2f}) pressure={p['pressure']:.3f} index={p['index']}")

## Step 4: Parse Point Data (coordinates, pressure, timestamps)

The points file format (binary):
- **Header**: Version (4 bytes) + Page ID (ASCII) + Points ID (ASCII)
- **Points**: Repeating structure of (X, Y, timestamp_delta, pressure, index)
  - X: 4 bytes (float32, big-endian)
  - Y: 4 bytes (float32, big-endian)  
  - Timestamp delta: 2-3 bytes
  - Pressure: 2 bytes
  - Index: 4 bytes (sequence number)

In [ ]:
def parse_shape_metadata(zf: zipfile.ZipFile) -> List[Dict]:
    """Extract stroke metadata from shape ZIP files."""
    strokes = []
    
    shape_files = [n for n in zf.namelist() if '/shape/' in n and n.endswith('.zip')]
    
    for shape_zip_path in shape_files:
        shape_data = zf.read(shape_zip_path)
        
        # Extract strings from the shape file
        shape_strings = extract_strings_from_binary(shape_data)
        
        # Parse JSON objects
        bbox = None
        metadata = None
        stroke_id = None
        points_id = None
        
        for s in shape_strings:
            if s.startswith('{') and 'bottom' in s and 'left' in s:
                bbox = json.loads(s)
            elif s.startswith('{') and 'dpi' in s:
                metadata = json.loads(s)
            elif len(s) == 36 and '-' in s:  # UUID format
                if stroke_id is None:
                    stroke_id = s
                elif points_id is None:
                    points_id = s
        
        if bbox:
            strokes.append({
                'id': stroke_id,
                'points_id': points_id,
                'bbox': bbox,
                'metadata': metadata
            })
    
    return strokes

with zipfile.ZipFile(note_file, 'r') as zf:
    strokes_meta = parse_shape_metadata(zf)
    
    print(f"=== Found {len(strokes_meta)} Strokes ===\n")
    for i, stroke in enumerate(strokes_meta, 1):
        print(f"Stroke {i}:")
        print(f"  ID: {stroke['id']}")
        print(f"  Points ID: {stroke['points_id']}")
        print(f"  Bounding Box: ({stroke['bbox']['left']:.1f}, {stroke['bbox']['top']:.1f}) → "
              f"({stroke['bbox']['right']:.1f}, {stroke['bbox']['bottom']:.1f})")
        if stroke['metadata']:
            print(f"  DPI: {stroke['metadata'].get('dpi', 'N/A')}")
            print(f"  Max Pressure: {stroke['metadata'].get('maxPressure', 'N/A')}")
        print()

## Step 3: Parse Shape Metadata (stroke bounding boxes)

In [ ]:
def extract_strings_from_binary(data: bytes) -> List[str]:
    """Extract null-terminated and length-prefixed strings from binary data."""
    strings = []
    i = 0
    while i < len(data):
        # Try to find JSON objects
        if data[i:i+1] == b'{':
            end = data.find(b'}', i)
            if end != -1:
                try:
                    json_str = data[i:end+1].decode('utf-8', errors='ignore')
                    json.loads(json_str)  # Validate
                    strings.append(json_str)
                    i = end + 1
                    continue
                except:
                    pass
        # Try ASCII strings
        if 32 <= data[i] <= 126:
            start = i
            while i < len(data) and (32 <= data[i] <= 126 or data[i] in (9, 10, 13)):
                i += 1
            if i - start > 3:
                strings.append(data[start:i].decode('ascii', errors='ignore'))
            continue
        i += 1
    return strings

with zipfile.ZipFile(note_file, 'r') as zf:
    # Find the note_info file
    note_info_path = [n for n in zf.namelist() if 'note/pb/note_info' in n][0]
    note_info_data = zf.read(note_info_path)
    
    strings = extract_strings_from_binary(note_info_data)
    
    print("=== Note Metadata ===\n")
    for s in strings[:5]:
        if len(s) > 100:
            print(f"{s[:100]}...")
        else:
            print(s)
        print()

## Step 2: Parse Note Metadata (note_info protobuf)

In [ ]:
note_file = Path("Notebook-1.note")
extract_dir = Path("notebook_extracted")

# List contents
with zipfile.ZipFile(note_file, 'r') as zf:
    print(f"Archive contains {len(zf.namelist())} files:\n")
    for name in sorted(zf.namelist())[:15]:
        info = zf.getinfo(name)
        print(f"  {info.file_size:>8} bytes - {name}")
    if len(zf.namelist()) > 15:
        print(f"  ... and {len(zf.namelist()) - 15} more files")

## Step 1: Extract and Examine the .note Archive

In [ ]:
import zipfile
import struct
import json
from pathlib import Path
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# For protobuf-like binary parsing
from io import BytesIO

# Onyx Boox .note File Parser & MyScript Converter

This notebook demonstrates how to parse native Onyx Boox `.note` files and prepare stroke data for handwriting recognition using MyScript API.

## Overview

Onyx `.note` files are ZIP archives containing:
- **Protobuf metadata** (`note/pb/note_info`) - document info, pen settings, page dimensions
- **Shape metadata** (`shape/*.zip`) - stroke bounding boxes, pen type, pressure settings  
- **Point data** (`point/**/*#points`) - binary-encoded stroke coordinates with pressure and timestamps

## Goal

Parse `Notebook-1.note` (3 ballpoint pen strokes) and convert to markdown using MyScript HWR engine.